In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key =  os.getenv('GROQ_API_KEY')

from langchain_groq import ChatGroq
model = ChatGroq(model = "openai/gpt-oss-20b",  groq_api_key = groq_api_key)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.14'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017E65643390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017E63CE7290>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates any input to Hindi, to the best to your ability"),
        MessagesPlaceholder(variable_name="messages")
    ])

chain = prompt | model

In [12]:
chain.invoke({'messages': [HumanMessage(content = "He went cold turkey")]})

AIMessage(content='वह ने अचानक से छोड़ दिया।', additional_kwargs={'reasoning_content': 'User says "He went cold turkey". They want translation to Hindi. According to instruction: translate any input to Hindi. So produce Hindi translation. "He went cold turkey" meaning he stopped abruptly. In Hindi: "वह अचानक से छोड़ गया" or "वह ने तुरंत छोड़ दिया". "He went cold turkey" literally: "वह ने अचानक छोड़ दिया" or "उसने अचानक से छोड़ दिया". Probably "वह ने अचानक से छोड़ दिया". Or "उसने अचानक से छोड़ दिया". Let\'s output Hindi.'}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 96, 'total_tokens': 222, 'completion_time': 0.198214322, 'completion_tokens_details': {'reasoning_tokens': 109}, 'prompt_time': 0.005289042, 'prompt_tokens_details': None, 'queue_time': 0.215891857, 'total_time': 0.203503364}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': '

In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#store all sessiosn
store = {}

def get_session_history(session_id: str)-> BaseChatMessageHistory:
    # create history wrt to the session id

    if session_id not in store:
        #whatever chat happens, we store it in the store's session id
        #here we load the previous chat history if it exists, and return it
        store[session_id]  = ChatMessageHistory()
        
    return store[session_id]

#get the entire chat history to the model. interact with model based on history
with_message_history = RunnableWithMessageHistory( chain, get_session_history)


In [19]:
config = {'configurable': {'session_id' : 'chat3'}}

response = with_message_history.invoke(
    [HumanMessage(content = "He went cold turkey")], 
    config=config
)

In [20]:
response.content

'उसने बिना तैयारी के अचानक छोड़ दिया।'

In [21]:
# add more complexity.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates any input to {input_language}, to the best to your ability"),
        MessagesPlaceholder(variable_name="messages")
    ])

chain = prompt | model

In [32]:
response = chain.invoke({'messages': [HumanMessage(content = "Transformers is a PyTorch-first library. It provides models that are faithful to their papers, easy to use, and easy to hack.")],
                         'input_language': 'French'
                         })
response.content

'Transformers est une bibliothèque centrée sur PyTorch. Elle propose des modèles fidèles à leurs articles, faciles à utiliser et simples à modifier.'

In [33]:
# now we will use multiple keys to save in the chat history

with_message_history = RunnableWithMessageHistory( chain, 
                                                  get_session_history,
                                                  input_messages_key = 'messages')

f:\projects_2026\Krishnayak - Langchain\ai_env\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [38]:
config = {'configurable': {'session_id' : 'chat4'}}

response = with_message_history.invoke(
    {"messages": [HumanMessage(content  = "Hi I am John. ")], 
     "input_language": "French"},
     config=config
)

response.content

'Bonjour, je suis John.'

In [39]:

response = with_message_history.invoke(
    {"messages": [HumanMessage(content  = "Whats my name ")], 
     "input_language": "hindi"},
     config=config
)
response.content

'मेरा नाम क्या है?'